# Tome C2, Chapter 8: Randomised Algorithms (Python companion)

Exercises first, solutions at the end. We run Karger's contraction algorithm and its recursive version and compare their success rates with the bounds of the chapter, test perfect matchings with random determinants, verify matrix products with Freivalds' trick, throw millions of balls into bins with one and two choices, and build three streaming sketches: Morris counters, count–min and the AMS estimator of $F_2$. We finish by derandomising a cut.

*Before each solution, predict what the output will look like.*

In [ ]:
import random, math, time, itertools
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
random.seed(8); rng = np.random.default_rng(8)

## Exercises

**P8.1 (★★) Minimum cuts.** Implement the contraction algorithm (Theorem 8.2) and the recursive contraction algorithm (Theorem 8.4). Measure how often each finds a minimum cut, against the bounds $2/(n(n-1))$ and $1/(2\log_2n+8)$, and check the values against `networkx.stoer_wagner`. Count the distinct minimum cuts found on a cycle (Corollary 8.3).

**P8.2 (★★) Algebra with coins.** Decide whether random graphs have a perfect matching with the Tutte matrix (Corollary 8.7) and compare with `networkx`; measure the error rate for small primes against the bound $n/p$. Time Freivalds' check (Proposition 8.8) against a matrix product.

**P8.3 (★★) Balls into bins.** Simulate $n$ balls in $n$ bins for $n$ up to a few million, with one, two and three choices, and compare the maximum loads with $\ln n/\ln\ln n$ and $\log_2\log_2n$ (Theorems 8.10 and 8.12).

**P8.4 (★★) Sketches.** Implement the universal family of Proposition 8.14 and check its collision rate; Morris counters (Theorem 8.15) and their averages; a count–min sketch (Theorem 8.16) on a Zipf-distributed stream; and the AMS estimator of $F_2$ (Theorem 8.17) with $4$-wise independent signs.

**P8.5 (★) Removing the coins.** Compare random cuts, the method of conditional expectations (Proposition 8.18) and the maximum cut on small random graphs; find by brute force a graph and an order on which the greedy rule is not optimal.

## Solutions

### P8.1 Minimum cuts
Contracting a uniformly random remaining edge, again and again, is the same as running through the edges in a uniformly random order and contracting each one whose ends are not yet merged (edges that became loops are skipped, which does not change the distribution of the next real contraction). With union–find, one run costs $O(m\,\alpha(n))$.

In [ ]:
def find(parent, u):
    while parent[u] != u:
        parent[u] = parent[parent[u]]; u = parent[u]
    return u

def contract_run(n, edges, target=2, rnd=random):
    order = edges[:]; rnd.shuffle(order)
    parent = list(range(n)); comps = n
    for u, v in order:
        if comps == target: break
        ru, rv = find(parent, u), find(parent, v)
        if ru != rv: parent[ru] = rv; comps -= 1
    side = [find(parent, u) for u in range(n)]
    return side

def cut_size(edges, side):
    return sum(side[u] != side[v] for u, v in edges)

def karger(n, edges, runs):
    best = None
    for _ in range(runs):
        side = contract_run(n, edges); c = cut_size(edges, side)
        if best is None or c < best[0]: best = (c, side)
    return best

def barbell(k, bridges):
    # two cliques of size k joined by `bridges` disjoint edges: the minimum cut separates them
    E = [(i, j) for i in range(k) for j in range(i + 1, k)]
    E += [(k + i, k + j) for i in range(k) for j in range(i + 1, k)]
    E += [(i, k + i) for i in range(bridges)]
    return 2 * k, E

for (n, E), name in [(barbell(6, 2), "barbell 6+6"), ((10, [(i, (i + 1) % 10) for i in range(10)] + [(0, 5), (2, 7)]), "cycle C10 + 2 chords")]:
    G = nx.Graph(); G.add_edges_from(E)
    k_true, _ = nx.stoer_wagner(G)
    T = 20000
    hits = sum(cut_size(E, contract_run(n, E)) == k_true for _ in range(T))
    print(f"{name:22s} n={n:2d}  min cut {k_true}:  success {hits / T:.4f}   bound 2/(n(n-1)) = {2 / (n * (n - 1)):.4f}")

On the cycle, every run ends with two arcs joined by two edges, so every run returns *a* minimum cut; but each particular one of the $\binom n2$ minimum cuts comes out with probability exactly $1/\binom n2$, and we should see them all.

In [ ]:
n = 8; E = [(i, (i + 1) % n) for i in range(n)]
seen = Counter()
for _ in range(20000):
    side = contract_run(n, E)
    seen[frozenset(frozenset(u for u in range(n) if side[u] == s) for s in set(side))] += 1
print(f"distinct minimum cuts found: {len(seen)}  (C(8,2) = {math.comb(8, 2)})")
print("frequencies:", sorted(round(c / 20000, 4) for c in seen.values())[:5], "...  target 1/28 =", round(1 / 28, 4))

The recursive version, on an explicit multigraph stored as a `Counter` of edge multiplicities. Contracting down to $t$ vertices picks edges with probability proportional to their multiplicity.

In [ ]:
def contract_to(mult, t, rnd=random):
    # mult: Counter {(u, v): multiplicity} with u < v, vertices are frozensets of original vertices
    mult = Counter(mult)
    verts = {x for e in mult for x in e}
    while len(verts) > t:
        edges = list(mult); w = [mult[e] for e in edges]
        u, v = rnd.choices(edges, weights=w)[0]
        new = u | v; verts -= {u, v}; verts.add(new)
        nm = Counter()
        for (a, b), c in mult.items():
            a2 = new if a in (u, v) else a; b2 = new if b in (u, v) else b
            if a2 != b2:
                key = (a2, b2) if sorted(a2) < sorted(b2) else (b2, a2)
                nm[key] += c
        mult = nm
    return mult

def brute_min_cut(mult):
    verts = sorted({x for e in mult for x in e}, key=sorted)
    best = None
    for mask in range(1, 2**(len(verts) - 1)):
        S = {verts[i] for i in range(len(verts)) if mask >> i & 1}
        c = sum(m for (a, b), m in mult.items() if (a in S) != (b in S))
        if best is None or c < best: best = c
    return best

def recursive_contraction(mult, rnd=random):
    nv = len({x for e in mult for x in e})
    if nv <= 6: return brute_min_cut(mult)
    t = math.ceil(1 + nv / math.sqrt(2))
    return min(recursive_contraction(contract_to(mult, t, rnd), rnd) for _ in range(2))

def as_mult(E):
    return Counter((frozenset([u]), frozenset([v])) if u < v else (frozenset([v]), frozenset([u])) for u, v in E)

rows = []
for k in (5, 7, 9, 12):
    n, E = barbell(k, 2)
    T = 200 if k < 12 else 80
    t0 = time.time(); ok = sum(recursive_contraction(as_mult(E)) == 2 for _ in range(T)); dt = (time.time() - t0) / T
    single = sum(cut_size(E, contract_run(n, E)) == 2 for _ in range(4000)) / 4000
    rows.append((n, ok / T, single, 1 / (2 * math.log2(n) + 8)))
    print(f"n={n:2d}  recursive: success {ok / T:.3f} ({dt * 1000:.0f} ms/run)   one contraction run: {single:.3f}   bound {rows[-1][3]:.3f}")

The recursive algorithm succeeds almost always on these graphs, far above its worst-case bound, and even a single contraction run succeeds a third to a half of the time, against a bound of a few percent. The bounds are worst cases over all graphs: for the barbell, most edges are inside the cliques, so the risky contractions are rarer than the proof allows.

### P8.2 Algebra with coins
Determinant modulo a prime by Gaussian elimination; the Tutte matrix gets uniform random entries.

In [ ]:
def det_mod(M, p):
    M = [row[:] for row in M]; n = len(M); d = 1
    for c in range(n):
        piv = next((r for r in range(c, n) if M[r][c] % p), None)
        if piv is None: return 0
        if piv != c: M[c], M[piv] = M[piv], M[c]; d = -d
        d = d * M[c][c] % p; inv = pow(M[c][c], p - 2, p)
        for r in range(c + 1, n):
            f = M[r][c] * inv % p
            if f: M[r] = [(a - f * b) % p for a, b in zip(M[r], M[c])]
    return d % p

def tutte_test(n, E, p, rnd=random):
    T = [[0] * n for _ in range(n)]
    for u, v in E:
        x = rnd.randrange(p); T[u][v] = x; T[v][u] = (-x) % p
    return det_mod(T, p) != 0

def has_pm(n, E):
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(E)
    return len(nx.max_weight_matching(G, maxcardinality=True)) * 2 == n

wrong = Counter(); total = Counter(); P = 1_000_003
for t in range(300):
    n = 20; q = random.choice([0.08, 0.12, 0.2])
    E = [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < q]
    truth = has_pm(n, E); total[truth] += 1
    wrong[truth] += tutte_test(n, E, P) != truth
print("graphs with / without a perfect matching:", total[True], total[False], "  wrong answers:", wrong[True], wrong[False])

With $p\approx10^6$ the test never errs here. With tiny primes, graphs that do have a perfect matching are sometimes reported without one (never the converse), and the bound $n/p$ of the Schwartz–Zippel lemma is the right order. For $p=2$ the test is useless: the matrix is then symmetric with zero diagonal, and over $\mathbb F_2$ such a matrix... we let the numbers speak.

In [ ]:
n = 12
graphs = []
while len(graphs) < 200:
    E = [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < 0.3]
    if has_pm(n, E): graphs.append(E)
for p in (2, 3, 5, 7, 11, 13, 31, 101, 1009):
    miss = np.mean([not tutte_test(n, E, p) for E in graphs for _ in range(5)])
    print(f"p = {p:5d}: false 'no' rate {miss:.3f}   bound n/p = {min(1, n / p):.3f}")

For the smallest primes the bound $n/p$ is at least $1$ and says nothing, although the test still catches some matchings. Corollary 8.7 asks for $p\ge2n$, and from there on the measured rate of false "no" sits well below $n/p$: the Schwartz–Zippel bound is a worst case over all polynomials of degree $n$, and determinants of Tutte matrices are far from the worst.

Freivalds' check against a full product, with `numpy` integer matrices:

In [ ]:
for n in (200, 400, 800):
    A = rng.integers(-9, 10, (n, n)); B = rng.integers(-9, 10, (n, n))
    t0 = time.time(); Cm = A @ B; t_prod = time.time() - t0
    Cbad = Cm.copy(); Cbad[rng.integers(n), rng.integers(n)] += 1
    t0 = time.time()
    caught = 0
    for _ in range(20):
        r = rng.integers(0, 2, n)
        caught += not np.array_equal(A @ (B @ r), Cbad @ r)
    t_check = (time.time() - t0) / 20
    print(f"n={n}: product {t_prod * 1000:7.1f} ms, one check {t_check * 1000:6.2f} ms, single wrong entry caught {caught}/20 times")

A single wrong entry $d_{ij}$ is caught exactly when $r_j=1$: half of the time, which is the bound of Proposition 8.8 attained. Ten checks bring the error below $10^{-3}$ for a fraction of the cost of one product.

### P8.3 Balls into bins
One choice is a histogram (`numpy.bincount`). For $d$ choices the balls must be placed one at a time, since each sees the loads left by the previous ones; a plain loop handles a few million balls.

In [ ]:
def max_load_one(n, rng):
    return int(np.bincount(rng.integers(0, n, n), minlength=n).max())

def max_load_d(n, d, rng):
    loads = [0] * n
    choices = rng.integers(0, n, (n, d)).tolist()
    for ch in choices:
        b = min(ch, key=loads.__getitem__); loads[b] += 1
    return max(loads)

ns = [10**3, 10**4, 10**5, 10**6, 3 * 10**6]
res = {1: [], 2: [], 3: []}
t0 = time.time()
for n in ns:
    reps = 5 if n <= 10**5 else 2
    res[1].append(np.mean([max_load_one(n, rng) for _ in range(reps)]))
    for d in (2, 3):
        res[d].append(np.mean([max_load_d(n, d, rng) for _ in range(reps if n < 10**6 else 1)]))
    print(f"n={n:>8d}: max load  one choice {res[1][-1]:4.1f}   two {res[2][-1]:3.1f}   three {res[3][-1]:3.1f}"
          f"   ln n/ln ln n = {math.log(n) / math.log(math.log(n)):4.2f}   log2 log2 n = {math.log2(math.log2(n)):4.2f}")
print(f"{time.time() - t0:.0f} s")

In [ ]:
x = np.array(ns, dtype=float)
plt.figure(figsize=(6, 3.4))
for d, mk in ((1, "o-"), (2, "s-"), (3, "^-")):
    plt.semilogx(x, res[d], mk, label=f"{d} choice" + ("s" if d > 1 else ""))
plt.semilogx(x, np.log(x) / np.log(np.log(x)), "k:", label="ln n / ln ln n")
plt.semilogx(x, np.log2(np.log2(x)), "k--", label="log2 log2 n")
plt.xlabel("n"); plt.ylabel("maximum load"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

The one-choice maximum grows slowly but steadily, well above $\ln n/\ln\ln n$, which is only the leading term: the corrections are of relative order $\ln\ln\ln n/\ln\ln n$, which decays so slowly that it is still large for every $n$ a computer can handle; two choices sit at $3$ or $4$ for every $n$ we can afford, and three choices gain much less than the second did. The theorem's constant $6$ is generous; the truth is $\log_2\log_2 n+O(1)$ with a small $O(1)$.

### P8.4 Sketches
The universal family of Proposition 8.14, with $p=2^{31}-1$: the collision rate of two fixed keys over random functions.

In [ ]:
PR = 2**31 - 1
def universal(w, rnd=random):
    a, b = rnd.randrange(1, PR), rnd.randrange(PR)
    return lambda x: ((a * x + b) % PR) % w

w = 50; T = 200000; x, y = 12345, 67890
coll = 0
for _ in range(T):
    h = universal(w); coll += h(x) == h(y)
print(f"collision rate {coll / T:.5f}   bound 1/w = {1 / w:.5f}")

Morris counters: one counter is a rough estimate; the average of $k$ independent counters has relative standard deviation about $1/\sqrt{2k}$, by Theorem 8.15. The register holds $X\approx\log_2n$, a number that fits in $5$ bits for a million events.

In [ ]:
def morris(n, rnd):
    X = 0
    for _ in range(n):
        if rnd.random() < 2.0**-X: X += 1
    return X

n = 100000; rnd = random.Random(1)
Xs = [morris(n, rnd) for _ in range(400)]
est = np.array([2**X - 1 for X in Xs], dtype=float)
print(f"n = {n}: mean of estimates {est.mean():.0f}, relative std of one counter {est.std() / n:.3f} (theory {math.sqrt((n - 1) / (2 * n)):.3f}), register values {min(Xs)}..{max(Xs)}")
for k in (1, 4, 16, 64):
    groups = est[: (400 // k) * k].reshape(-1, k).mean(axis=1)
    print(f"  average of {k:2d} counters: relative std {groups.std() / n:.3f}   theory {1 / math.sqrt(2 * k):.3f}")

Count–min on a stream of $10^6$ items drawn from a Zipf law on $10^5$ keys (frequency of the $r$th key proportional to $1/r^{1.1}$). With $\varepsilon=0.001$ and $\delta=0.01$ we need $w=\lceil e/\varepsilon\rceil=2719$ and $d=5$: about $13\,600$ counters instead of $10^5$.

In [ ]:
class CountMin:
    def __init__(self, eps, delta, rnd):
        self.w = math.ceil(math.e / eps); self.d = math.ceil(math.log(1 / delta))
        self.ab = [(rnd.randrange(1, PR), rnd.randrange(PR)) for _ in range(self.d)]
        self.T = np.zeros((self.d, self.w), dtype=np.int64)
    def cols(self, keys):
        return [((a * keys + b) % PR) % self.w for a, b in self.ab]
    def add_many(self, keys):
        for j, c in enumerate(self.cols(keys)):
            np.add.at(self.T[j], c, 1)
    def query(self, keys):
        return np.min([self.T[j][c] for j, c in enumerate(self.cols(keys))], axis=0)

U, N = 10**5, 10**6
probs = 1 / np.arange(1, U + 1)**1.1; probs /= probs.sum()
stream = rng.choice(U, size=N, p=probs).astype(np.int64)
true = np.bincount(stream, minlength=U)
cm = CountMin(0.001, 0.01, random.Random(3)); cm.add_many(stream)
keys = np.arange(U, dtype=np.int64); est = cm.query(keys)
err = est - true
print(f"counters: {cm.d} x {cm.w} = {cm.d * cm.w};  never underestimates: {bool((err >= 0).all())}")
print(f"fraction of keys with error >= eps*N = {0.001 * N:.0f}: {(err >= 0.001 * N).mean():.5f}  (bound delta = 0.01)")
print("top 5 keys, true vs estimated:", list(zip(true[:5].tolist(), est[:5].tolist())))

The AMS sketch: $4$-wise independent signs from a random cubic polynomial over $\mathbb F_{2^{61}}$ would follow the chapter exactly; for speed we use the prime field $\mathbb F_p$, $p=2^{61}-1$, and the parity of the value, whose bias ($1/p$) is far below anything we can measure. We estimate $F_2$ of the same stream.

In [ ]:
P61 = 2**61 - 1
def sign_family(rnd):
    c = [rnd.randrange(P61) for _ in range(4)]
    def s(keys):
        v = np.zeros_like(keys, dtype=object)
        k = keys.astype(object)
        for coef in reversed(c): v = (v * k + coef) % P61
        return np.where(np.array([int(t) & 1 for t in v]) == 1, 1, -1)
    return s

F2 = int((true.astype(np.int64)**2).sum())
rnd = random.Random(5)
Zs = []
for _ in range(240):
    s = sign_family(rnd)
    Zs.append(int((s(keys) * true).sum())**2)
Zs = np.array(Zs, dtype=float)
print(f"true F2 = {F2:.4e},  mean of Z^2 = {Zs.mean():.4e},  relative std of one Z^2 = {Zs.std() / F2:.3f} (bound sqrt 2 = 1.414)")
for k in (1, 4, 15):
    meds = []
    for rep in range(200):
        idx = np.random.default_rng(rep).permutation(len(Zs))[: 8 * k]
        meds.append(np.median(Zs[idx].reshape(8, k).mean(axis=1)))
    meds = np.array(meds)
    print(f"  median of 8 averages of {k:2d}: relative error 90% quantile {np.quantile(abs(meds / F2 - 1), 0.9):.3f}")

(The sketch was computed here from the final frequencies, $Z=\sum_xs(x)f_x$, which is what a streaming update $Z\leftarrow Z+s(x)c$ produces at the end.) One $Z^2$ is unbiased but noisy; averages of $k$ copies shrink the error like $1/\sqrt k$, and the median of several averages protects against the occasional bad one, as in Theorem 8.17.

### P8.5 Removing the coins

In [ ]:
def greedy_cut(n, E, order):
    side = {}; adj = {u: [] for u in range(n)}
    for u, v in E: adj[u].append(v); adj[v].append(u)
    for v in order:
        placed = [side[u] for u in adj[v] if u in side]
        side[v] = 1 if placed.count(0) > placed.count(1) else 0   # opposite to the majority, ties to side 0
    return sum(side[u] != side[v] for u, v in E)

def max_cut(n, E):
    return max(sum(((mask >> u) ^ (mask >> v)) & 1 for u, v in E) for mask in range(2**(n - 1)))

stats = []
for t in range(200):
    n = 10; E = [(u, v) for u in range(n) for v in range(u + 1, n) if random.random() < 0.4]
    if not E: continue
    m = len(E); g = greedy_cut(n, E, list(range(n))); opt = max_cut(n, E)
    stats.append((m / 2 / opt, g / opt))
s = np.array(stats)
print(f"200 random graphs on 10 vertices: random cut (expected) / max = {s[:, 0].mean():.3f},  greedy / max = {s[:, 1].mean():.3f} (min {s[:, 1].min():.3f})")

# smallest example where some order makes the greedy rule miss the maximum cut
found = None
for n in range(2, 6):
    pairs = list(itertools.combinations(range(n), 2))
    for k in range(1, len(pairs) + 1):
        for E in itertools.combinations(pairs, k):
            opt = max_cut(n, E)
            for order in itertools.permutations(range(n)):
                if greedy_cut(n, list(E), order) < opt:
                    found = (n, E, order, greedy_cut(n, list(E), order), opt); break
            if found: break
        if found: break
    if found: break
print("smallest bad example: n, edges, order, greedy, max =", found)

The greedy cut is within a few percent of the maximum on random graphs, much better than its guarantee of $m/2$; but it can be fooled already on a path with three edges, as in Exercise 8.10. Finding a maximum cut is NP-hard (Tome C3), and Chapter 12 will do better than $\tfrac12$ with Goemans and Williamson's random hyperplane.